In [ ]:
%%capture
# Installations 
!pip install pandas numpy scikit-learn nltk spacy matplotlib seaborn wordcloud --quiet

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('vader_lexicon')

!python -m spacy download en_core_web_sm

! pip install pandas numpy scikit-learn nltk spacy matplotlib seaborn wordcloud --quiet

%%capture
# Download NLTK resources
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('vader_lexicon')

# Download spaCy model
!python -m spacy download en_core_web_sm

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [10]:
# Imports
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# NLP
import re
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer


# Modeling
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [5]:
# Load the dataset

df = pd.read_csv('../data/all_data.csv')
df.head()

,Product Name,Price,Review
0,Acer Aspire 3 A315-24P AMD Ryzen 3 7320U 8GB R...,"Tk 53,060",I Bought this one about one year ago.\r\nIts a...
1,Lenovo IdeaPad 1 15AMN7 AMD Ryzen 3 7320U 8GB ...,"Tk 54,790",Overall good deal as per price.
2,Lenovo IdeaPad 1 15AMN7 AMD Ryzen 3 7320U 8GB ...,"Tk 54,790",Hope the quality of this product will be very ...
3,Lenovo IdeaPad 1 15AMN7 AMD Ryzen 3 7320U 8GB ...,"Tk 54,790",Great affordable performance
4,Lenovo IdeaPad 1 15AMN7 AMD Ryzen 3 7320U 8GB ...,"Tk 54,790",Good product


In [6]:
# Initialize tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    tokens = nltk.word_tokenize(text)
    
    cleaned_tokens = [
        lemmatizer.lemmatize(word) 
        for word in tokens 
        if word not in stop_words and len(word) > 2
    ]
    
    return " ".join(cleaned_tokens)

df['clean_text'] = df['Review'].apply(clean_text)
df[['Review', 'clean_text']].head()

,Review,clean_text
0,I Bought this one about one year ago.\r\nIts a...,bought one one year ago good laptop basic work...
1,Overall good deal as per price.,overall good deal per price
2,Hope the quality of this product will be very ...,hope quality product good
3,Great affordable performance,great affordable performance
4,Good product,good product


In [12]:
sia = SentimentIntensityAnalyzer()

def get_sentiment(text):
    text = str(text)  # force string
    
    if text.strip() == "":
        return 'neutral'
    
    score = sia.polarity_scores(text)['compound']
    
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['sentiment'] = df['Review'].apply(get_sentiment)
df['sentiment'].value_counts()
   

sentiment
positive    20838
neutral      4729
negative      324
Name: count, dtype: int64

In [13]:
# Train-test split

X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [14]:
nb_pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

nb_pipeline.fit(X_train, y_train)

y_pred = nb_pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.8824097316084186
              precision    recall  f1-score   support

    negative       0.58      0.17      0.26        65
     neutral       0.92      0.45      0.61       946
    positive       0.88      0.99      0.93      4168

    accuracy                           0.88      5179
   macro avg       0.79      0.54      0.60      5179
weighted avg       0.88      0.88      0.86      5179

